# Overview

# Pretraining Your RQwen3 Model: A Complete Step-by-Step Walkthrough

---

## What Is Pretraining?

Before we touch any code, let's understand what we're actually doing.

Your RQwen3 model right now has **randomly initialized weights**. Every matrix in every layer — the attention projections, the feedforward networks, the embeddings — is filled with random numbers. That's why when you ran it, you got:

```
slidمعالجةפוטacedصب갔/apisliked渤咣 valued
```

The model is literally guessing random tokens. **Pretraining** is the process of teaching the model to predict language by showing it massive amounts of text and adjusting those random weights until the model learns patterns of language.

The task is simple: **next-token prediction**. Given a sequence of tokens, predict what comes next.

```
Input:  "The cat sat on the"
Target: "cat sat on the mat"
```

Every token in the sequence serves as both context (for predicting future tokens) and a target (to be predicted from past tokens). The model learns by getting billions of these examples wrong, computing how wrong it was, and nudging its weights to be less wrong next time.

---

## Step 0: Understand the Big Picture

Here's the full pipeline at a glance:

```
Raw Text Data
     │
     ▼
[Tokenizer] ──► Converts text to integer token IDs
     │
     ▼
[Chunking] ──► Splits token streams into fixed-length sequences (2048 tokens)
     │
     ▼
[DataLoader] ──► Batches sequences together for efficient GPU processing
     │
     ▼
[Training Loop]
     │
     ├── Forward Pass: Model predicts next token at every position
     ├── Loss Calculation: How wrong were the predictions?
     ├── Backward Pass: Compute gradients (which direction to nudge weights)
     └── Optimizer Step: Actually nudge the weights
     │
     ▼
[Repeat millions of times]
     │
     ▼
Model that understands language!
```

#### **Summary: The Complete Conceptual Flow**

```
                        "Once upon a time there was"
                                    │
                            [Tokenizer]
                                    │
                        [531, 2089, 264, 882, 1052, 574]
                                    │
            ┌───────────────────────┴───────────────────────┐
            │                                               │
      Input (x):                                     Target (y):
  [531, 2089, 264, 882, 1052]                [2089, 264, 882, 1052, 574]
            │                                               │
      [RQwen3 Model]                                        │
            │                                               │
  Logits: predictions for                                   │
  each position's next token                                │
            │                                               │
            └──────────── Cross-Entropy Loss ───────────────┘
                                    │
                              [Backpropagation]
                                    │
                          Gradients for every weight
                                    │
                           [AdamW Optimizer]
                                    │
                        Weights nudged slightly
                                    │
                         [Repeat millions of times]
                                    │
                        Model that understands language!
```

---
# Initial Steps

## Imports & Path Management

In [ ]:
import sys
sys.path.insert(0, '..') # Add parent directory to sys.path to import src (My Package)

In [ ]:
import os
import torch
import numpy as np
import torch
import torch.nn as nn
from src import *
from dataclasses import dataclass
from typing import Union
from abc import abstractmethod
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import Markdown
from datasets import load_dataset

## Configurations

In [ ]:
# Initial model configurations for my model RQwen3
model_config_dict: dict[str, Union[int, float, torch.dtype]] = {
    "d_model": 512,
    "dtype": torch.float32,
    "rms_norm_eps": 1e-6,
    "dropout": 0.1,
    "max_seq_len": 2048,
    "vocab_size": 151_936,
    "n_layer": 28,
    "num_heads": 16,
    "num_kv_heads": 8,
    "head_dim": 128,
    "intermediate_size": 6144,
    "rope_theta": 1_000_000.0
}

INPUT_TEXT = "Why was the Transformer model architecture so important to ML?"
DEVICE = get_device()

os.environ["HF_DATASETS_CACHE"] = "./data/hf_cache"

## Model Loading

In [ ]:
print('Loading config and my model...')
MODEL_CONFIG = CoreConfig(model_config_dict)
rqwen3 = RQwen3(MODEL_CONFIG).to(DEVICE)
print('Loaded RQwen3 model.')

In [ ]:
print('Loading trained Qwen3-1.7B...')
qwen3_1_7B = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B",
    torch_dtype=torch.float16,
    device_map='auto',
    attn_implementation='eager',  # required to extract attention weights
)
print('Loaded Qwen3-1.7B model.')

---
# Model Training

> For a causal language model like yours, pretraining is next-token prediction. The loss is cross-entropy between the model's predicted logit distribution and the actual next token at every position.

In [ ]:
"""Training utilities: config, checkpointing, snapshots, and generation."""

@dataclass
class TrainConfig:
    """Training hyperparameters and paths."""
    # Data
    dataset_name: str = 'HuggingFaceFW/fineweb-edu'
    seq_len: int = 2048

    # Training
    batch_size: int = 4
    grad_accum_steps: int = 8
    max_steps: int = 10_000
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    warmup_steps: int = 500
    max_grad_norm: float = 1.0

    # Logging & checkpoints
    log_every: int = 10
    save_every: int = 1000
    sample_every: int = 500

    # Paths
    checkpoint_dir: str = '../checkpoints'
    snapshot_dir: str = '../snapshots'


def save_checkpoint(model, optimizer, scheduler, step, loss_history, path):
    """Save everything needed to resume training."""
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss_history': loss_history,
    }, path)
    print(f'  Checkpoint saved: {path}')


def load_checkpoint(path, model, optimizer, scheduler):
    """Resume training from a checkpoint."""
    device = DEVICE
    ckpt = torch.load(path, weights_only=False, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    print(f'Resumed from step {ckpt["step"]}')
    return ckpt['step'], ckpt['loss_history']


def save_snapshot(model, name, snapshot_dir):
    """Save weight stats snapshot (compatible with notebook 03/04 comparison tools)."""
    snapshot = {}
    for pname, param in model.named_parameters():
        data = param.detach().cpu().float()
        snapshot[pname] = {
            'mean': data.mean().item(),
            'std': data.std().item(),
            'min': data.min().item(),
            'max': data.max().item(),
            'shape': list(data.shape),
            'histogram': np.histogram(data.numpy().flatten(), bins=50),
        }
    path = os.path.join(snapshot_dir, f'{name}.pt')
    torch.save(snapshot, path)
    return path


@torch.no_grad()
def generate_sample(model, tokenizer, prompt, max_new_tokens=60):
    """Simple greedy generation to monitor training progress."""
    device = get_device()
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    for _ in range(max_new_tokens):
        logits = model(ids)
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    model.train()
    return tokenizer.decode(ids[0], skip_special_tokens=True)


---
# Testing Loaded Model

In [ ]:
class SendToModel:
    def __init__(self, model):
        self.model = model
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
        self.is_hf_model = hasattr(model, 'generate')

    def __call__(self, inputs, max_new_tokens=100):
        tokenized_inputs = self.tokenizer(inputs, return_tensors='pt', padding=True).to(DEVICE)

        if self.is_hf_model:
            output_ids = self.model.generate(**tokenized_inputs, max_new_tokens=max_new_tokens)
            return self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
        else:
            # Single forward pass for your custom model
            output = self.model(tokenized_inputs.input_ids)
            # Handle if output is a tuple/object with logits or raw tensor
            logits = output.logits if hasattr(output, 'logits') else output
            predicted_ids = logits.argmax(dim=-1)
            return self.tokenizer.decode(predicted_ids[0], skip_special_tokens=True)


Message_Qwen3 = SendToModel(qwen3_1_7B)
Message_RQwen3 = SendToModel(rqwen3)

response_qwen3 = Message_Qwen3(INPUT_TEXT)
response_rqwen3 = Message_RQwen3(INPUT_TEXT)

In [ ]:
display(Markdown(f"**Qwen3 Response:** {response_qwen3}"))
display(Markdown(f"**RQwen3 Response:** {response_rqwen3}"))

---
# PreTrain RQwen3
> The core goal of pretraining is to teach the model language structure, world knowledge, and reasoning patterns through next-token prediction. 

#### In order to do this properly wee need:  
-   **Diverse in domain and register:** 
The model needs to handle casual conversation, technical explanations, structured tasks, and formal writing. A model trained only on Wikipedia will be articulate but stiff and bad at dialogue. One trained only on Reddit will be conversational but shallow on facts. So we want both — plus code, books, academic text, etc.

- **High signal-to-noise ratio:**
Raw web scrapes (like Common Crawl) are enormous but dirty — lots of spam, SEO garbage, duplicate boilerplate, and malformed text. Good pretraining data has been filtered. For example, C4 (Colossal Clean Crawled Corpus) is Common Crawl that's been aggressively cleaned by Google: deduped, filtered for natural language, removed pages with bad words/short lines/etc. The difference between raw Common Crawl and C4 in downstream quality is measurable.

-   **Appropriate token length distribution:** 
You want documents long enough to teach long-range dependencies and context, but your dataset should also have short documents so the model learns to handle varied context windows. The Pile handles this naturally due to its mixture of sources.

- **Low duplication:**
Duplicated data causes the model to memorize rather than generalize. Research from Google (the "Deduplicating Training Data Makes Language Models Better" paper) showed that even near-duplicate removal significantly improves perplexity and reduces memorization.

- **Domain relevance to your use case:**
Since RQwen3 is a tutor + personal manager, we specifically want strong representation of:
    - Instructional/explanatory text — textbooks, tutorials, how-to guides (look at OpenAssistant, Dolly, or even the textbook subsets in The Pile)
    - Structured reasoning — math, logic, code (DM Mathematics in The Pile, GitHub subset, ArXiv)
    - Conversational/dialogue text — important for the chat assistant behavior (though this bleeds into fine-tuning territory)

#### The Datasets being used are:
Let me look up those datasets you linked first before building the table.Good, got the details. Now let me check a few of the classic datasets too for accuracy on sizes/links.Good, got everything. Now let me build the full reference table and training guide.

---

####  **Pretraining Dataset Reference Table**

| **Name** | **Size** | **Categories / About** | **Link** | **Best Pretraining Usage** |
|---|---|---|---|---|
| **The Pile** | ~825 GB (~300B tokens) | 22 subsets: books, web, GitHub, ArXiv, FreeLaw, PubMed, Wikipedia, DM Math, StackExchange, and more | [HuggingFace](https://huggingface.co/datasets/EleutherAI/pile) | The gold standard general-purpose pretraining corpus. Best for building broad world knowledge, language diversity, and covering many domains simultaneously. Your primary backbone dataset. |
| **FineWeb-Edu** | ~1.3T tokens (also a 5.4T version) | Web pages filtered by an LLaMA-3-70B educational quality classifier — only high-scoring educational content makes it in | [HuggingFace](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) | Exceptional for tutor/assistant use cases. Teaches explanatory writing, structured reasoning, and knowledge-dense text. Upsample this heavily given your use case. One of the best modern pretraining sources available. |
| **FineFineWeb** | ~varies by domain subset (large, domain-sliced) | Domain-specific slices of web data across aerospace, agronomy, art, law, medicine, etc. — fine-grained domain web corpus | [HuggingFace](https://huggingface.co/datasets/m-a-p/FineFineWeb) | Best for targeted domain coverage. If you want the model to be strong in specific knowledge areas (e.g., science, law, medicine), pull the relevant domain slices. Treat it as a precision tool alongside broader corpora. |
| **C4** (Colossal Clean Crawled Corpus) | ~305 GB English (~156B tokens) | Aggressively cleaned Common Crawl web text — filtered for natural language, deduped, bad content removed | [HuggingFace](https://huggingface.co/datasets/allenai/c4) | Foundational clean web text. Great for teaching natural prose, general language patterns, and common knowledge. Well-studied and reliable. Good as your "baseline web text" source. |
| **RedPajama-1T** | ~1.2T tokens | 7 subsets mirroring LLaMA's training mix: Common Crawl, C4, GitHub, Books, ArXiv, Wikipedia, StackExchange | [HuggingFace](https://huggingface.co/datasets/togethercomputer/RedPajama-Data-1T) | A complete pretraining-ready mixture at massive scale. If you want one dataset that already has a balanced mix baked in, this is it. Strong alternative to hand-rolling your own mixture from scratch. |
| **Wikipedia** (Wikimedia) | ~20 GB English (~4B tokens), 300+ languages | Full cleaned Wikipedia articles across all languages, one subset per language | [HuggingFace](https://huggingface.co/datasets/wikimedia/wikipedia) | Factual grounding and structured knowledge. Always upsample this relative to its natural size — it's high-signal. Critical for making your model reliably knowledgeable. |
| **GitHub Code** (CodeParrot) | ~1 TB | 115M code files, 32 programming languages | [HuggingFace](https://huggingface.co/datasets/codeparrot/github-code) | Code understanding and structured reasoning. Even for a non-code assistant, code pretraining measurably improves logical reasoning and structured output. Include at ~10-15% of your mixture. |
| **Lucie Training Dataset** | ~10T tokens | Multilingual: English, French, German, Spanish, Italian + code. Sources: web, subtitles, academic papers, books, newspapers | [HuggingFace](https://huggingface.co/datasets/OpenLLM-France/Lucie-Training-Dataset) | Best if you want multilingual capability baked in at pretraining. Also useful for its diverse source mix (subtitles and newspapers add register diversity most corpora lack). Skip if English-only is your target. |



#### **Recommended Sampling Weights for Your Use Case**

Given your goal (general chat assistant + knowledgeable tutor), here's a concrete starting mixture:

| Source | Suggested Weight | Reasoning |
|---|---|---|
| FineWeb-Edu | **30%** | Your biggest upsample — directly trains the tutoring/explanatory voice |
| The Pile (mixed) | **20%** | Broadest diversity, covers domains nothing else touches |
| C4 / RedPajama web | **15%** | General natural language, common knowledge |
| Wikipedia | **12%** | Factual grounding, upsample 3-4x its natural size |
| Books (Books3 / Pile subset) | **12%** | Long-form coherence, reasoning across paragraphs |
| GitHub Code | **8%** | Structural reasoning benefits |
| FineFineWeb (domain slices) | **3%** | Targeted boosts in specific knowledge areas |


## Loading & Processing Datasets

### TinyStories

In [ ]:
# Datasets Used for Pre-Training (NLP)
ds_tiny_stories = load_dataset("roneneldan/TinyStories", split="train[:2%]", cache_dir="../data/tiny_stories")

# Load slices (streaming=False downloads to cache, slice notation limits size)
c4 = load_dataset("allenai/c4", "en", split="train[:2%]", cache_dir="../data/c4")
arxiv = load_dataset("togethercomputer/RedPajama-Data-1T", "arxiv", split="train[:2%]", cache_dir="../data/arxiv")
github = load_dataset("bigcode/the-stack", data_dir="../data/python", split="train[:2%]", cache_dir="../data/github")

### The Pile
> The Pile is a strong choice. It's a 825GB dataset curated by EleutherAI specifically for LLM pretraining, composed of 22 subsets including Books3, OpenWebText2, ArXiv, GitHub, FreeLaw, PubMed, Wikipedia, DM Mathematics, and more. Each subset was chosen to cover a different slice of human knowledge and writing style.